### Dependencias
'''
pip install langchain langchain-openai langchain-community chromadb tiktoken
'''

In [11]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [4]:
# 1. PREPARAR EL DOCUMENTO DE PRUEBA
# Creamos un archivo de texto simulando información interna de tu app
conocimiento_interno = """
Protocolo de Ansiedad - Organización Alfa:
Cuando un usuario presente síntomas de ansiedad leve, el primer paso es guiarlo en una respiración 4-7-8.
No se deben recomendar medicamentos en esta fase. El contacto de emergencia de la red es emergency services.
"""

with open("protocolo_alfa.txt", "w", encoding="utf-8") as f:
    f.write(conocimiento_interno)

print("1. Documento de conocimiento creado.")

1. Documento de conocimiento creado.


In [5]:
# 2. CARGAR Y PICAR EL TEXTO (Chunking)
loader = TextLoader("protocolo_alfa.txt", encoding="utf-8")
documentos = loader.load()

# Picamos el texto en fragmentos pequeños para que la base de datos vectorial los indexe mejor
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
fragmentos = text_splitter.split_documents(documentos)

print(f"2. Texto dividido en {len(fragmentos)} fragmento(s).")

2. Texto dividido en 1 fragmento(s).


In [6]:
# 3. CREAR EMBEDDINGS Y GUARDAR EN CHROMA (Base de Datos Vectorial)
# Transformamos el texto en números (vectores) usando el modelo oficial de OpenAI [3]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Inicializamos Chroma en memoria (no creará archivos locales, ideal para pruebas)
vector_store = Chroma.from_documents(fragmentos, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 1}) # Recupera el fragmento más cercano

print("3. Fragmentos convertidos a vectores y guardados en Chroma.")

3. Fragmentos convertidos a vectores y guardados en Chroma.


In [8]:
# 4. CONFIGURAR EL PROMPT Y EL MODELO DE LENGUAJE (LLM)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Diseñamos las instrucciones para el modelo (System Prompt)
system_prompt = (
    "Eres un asistente virtual de ayuda psicológica. "
    "Responde la pregunta del usuario utilizando únicamente el siguiente contexto proporcionado. "
    "Si no sabes la respuesta, di que no la conoces.\n\n"
    "Contexto:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

In [9]:
# 5. ORQUESTAR EL RAG CON LANGCHAIN
# Unimos el recuperador de Chroma con el generador de OpenAI
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("4. Sistema RAG listo y acoplado.\n" + "="*40)

4. Sistema RAG listo y acoplado.


In [10]:
# 6. HACER UNA PREGUNTA (Simulación de la App Móvil)
pregunta_usuario = "¿Qué debo hacer si un usuario tiene ansiedad leve según el protocolo?"
print(f"Pregunta del usuario: {pregunta_usuario}\n")

# Ejecutamos el flujo completo
respuesta = rag_chain.invoke({"input": pregunta_usuario})

print("Respuesta del RAG:")
print(respuesta["answer"])

Pregunta del usuario: ¿Qué debo hacer si un usuario tiene ansiedad leve según el protocolo?

Respuesta del RAG:
Si un usuario presenta síntomas de ansiedad leve, el primer paso es guiarlo en una respiración 4-7-8. No se deben recomendar medicamentos en esta fase.
